# M1' 修正版 · YOLOv11n + EMA@颈部P3（Colab）

对 M1 失败的两处修正：
1. **权重重映射**：预训练颈部/头部权重对齐加载，只有 EMA 本身是新层
2. **EMA 挪位**：主干末端（与 C2PSA 堆叠）→ 颈部 P3（小目标分支）

另含**数据完整性检查**（近重复帧跨集泄漏，约 1 分钟）。
操作同前：T4 → 全部运行 → 约 2 小时 → 下载 results.zip。

In [ ]:
!nvidia-smi

In [ ]:
# 下载数据集（同前）
API_KEY = 'YOUR_ROBOFLOW_API_KEY'

import json, glob, zipfile, urllib.request, subprocess

meta = json.load(urllib.request.urlopen(
    f'https://api.roboflow.com/km-sd0ce/pig-behavior-wlvku/1/yolov8?api_key={API_KEY}'))
subprocess.run(['curl', '-sL', '-o', '/content/dataset.zip', meta['export']['link']], check=True)
with zipfile.ZipFile('/content/dataset.zip') as z:
    z.extractall('/content/dataset')
DATA_YAML = glob.glob('/content/dataset/**/data.yaml', recursive=True)[0]
print('data.yaml:', DATA_YAML)

In [ ]:
# 数据严谨性检查：近重复帧跨集泄漏（感知哈希，抽样 500 张）
import glob, random
from PIL import Image

def ahash(path, size=16):
    img = Image.open(path).convert('L').resize((size, size))
    px = list(img.getdata())
    avg = sum(px) / len(px)
    return int(''.join('1' if p > avg else '0' for p in px), 2)

hashes = {}
for sp in ('train', 'valid', 'test'):
    files = glob.glob(f'/content/dataset/*/{sp}/images/*.*')
    hashes[sp] = {f: ahash(f) for f in files}
    print(sp, len(files))

random.seed(0)
sample = random.sample(list(hashes['train'].items()), min(500, len(hashes['train'])))
dups = 0
for sp in ('valid', 'test'):
    for _, h1 in sample:
        for _, h2 in hashes[sp].items():
            if (h1 ^ h2).bit_count() <= 10:
                dups += 1
print(f'疑似跨集近重复对: {dups}')
print('解读: 0~个位数 = 可接受；几十以上 = 存在泄漏，需重切分（把数字反馈给助手）')

In [ ]:
!pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# EMA 定义 + 注册 + 改进 yaml + 权重重映射 + 结构自检
import re, torch
from torch import nn
import ultralytics.nn.tasks as tasks

class EMA(nn.Module):
    """Efficient Multi-Scale Attention (ICASSP 2023)，输入输出通道数不变。"""
    def __init__(self, channels, factor=8):
        super().__init__()
        self.groups = factor
        assert channels // self.groups > 0
        self.softmax = nn.Softmax(-1)
        self.agp = nn.AdaptiveAvgPool2d((1, 1))
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.gn = nn.GroupNorm(channels // self.groups, channels // self.groups)
        self.conv1x1 = nn.Conv2d(channels // self.groups, channels // self.groups, 1)
        self.conv3x3 = nn.Conv2d(channels // self.groups, channels // self.groups, 3, padding=1)

    def forward(self, x):
        b, c, h, w = x.size()
        group_x = x.reshape(b * self.groups, -1, h, w)
        x_h = self.pool_h(group_x)
        x_w = self.pool_w(group_x).permute(0, 1, 3, 2)
        hw = self.conv1x1(torch.cat([x_h, x_w], dim=2))
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x1 = self.gn(group_x * x_h.sigmoid() * x_w.permute(0, 1, 3, 2).sigmoid())
        x2 = self.conv3x3(group_x)
        x11 = self.softmax(self.agp(x1).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x12 = x2.reshape(b * self.groups, c // self.groups, -1)
        x21 = self.softmax(self.agp(x2).reshape(b * self.groups, -1, 1).permute(0, 2, 1))
        x22 = x1.reshape(b * self.groups, c // self.groups, -1)
        weights = (torch.matmul(x11, x12) + torch.matmul(x21, x22)).reshape(b * self.groups, 1, h, w)
        return (group_x * weights.sigmoid()).reshape(b, c, h, w)

tasks.EMA = EMA

YAML_TEXT = r'''
nc: 10
scales:
  n: [0.50, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]
  - [-1, 1, EMA, [64]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]
  - [[17, 20, 23], 1, Detect, [nc]]
'''
with open('/content/yolo11-ema-neck-n.yaml', 'w') as f:
    f.write(YAML_TEXT.strip() + '\n')

from ultralytics import YOLO
model = YOLO('/content/yolo11-ema-neck-n.yaml')

# 权重重映射：插入点(17)之后的老层号 +1，使预训练颈部/头部对齐
from ultralytics.utils.downloads import attempt_download_asset
ckpt = torch.load(attempt_download_asset('yolo11n.pt'), map_location='cpu', weights_only=False)
sd = ckpt['model'].state_dict() if 'model' in ckpt else ckpt
remapped = {}
for k, v in sd.items():
    m = re.match(r'model\.(\d+)\.', k)
    if m and 17 <= int(m.group(1)) <= 23:
        k = k.replace(f'model.{m.group(1)}.', f'model.{int(m.group(1))+1}.', 1)
    remapped[k] = v
model_sd = model.model.state_dict()
remapped = {k: v for k, v in remapped.items() if k in model_sd and model_sd[k].shape == v.shape}
missing, unexpected = model.model.load_state_dict(remapped, strict=False)
print(f'权重迁移：成功 {len(remapped)} 键；缺失 {len(missing)} 键（应为 EMA 新层 + Detect 分类头），多余 {len(unexpected)} 键')

with torch.no_grad():
    _ = model.model.eval()(torch.zeros(1, 3, 640, 640))
print('结构自检通过')

In [ ]:
# 训练 M1'（同参：100 轮 / 640 / batch16）
model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
            device=0, project='/content/results', name='m1p-ema-neck')

In [ ]:
# 评估 + 保存指标
import json

metrics = model.val()
summary = {
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50-95': round(float(metrics.box.map), 4),
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
}
with open('/content/results/m1p-ema-neck/metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(summary)
print('对照基线: mAP50=0.5706  mAP50-95=0.4169  P=0.5132  R=0.5868')
print('对照 M1 : mAP50=0.5411（EMA@主干末端，已否决）')

In [ ]:
# 打包下载
import shutil
from google.colab import files

shutil.make_archive('/content/results', 'zip', '/content/results')
files.download('/content/results.zip')